# BloomML — R2 real CUDA physics on 2× T4

**Research only.** Runs the new **tensorized thermodynamic solver** on both separate T4 cards. CPU scalar solver is the numerical reference. This is not calibrated physics or production training.

Kaggle: private notebook; GPU T4 x2; Internet ON. Outputs remain private. Fresh session for Git clone.


In [ ]:
import subprocess
from pathlib import Path
REPOSITORY = "https://github.com/MichalMatu/bloomml.git"
BRANCH = "research/simulator-fidelity-kaggle-20261103"
SOURCE = Path("/kaggle/working/bloomml-r2-source")
if SOURCE.exists():
    raise RuntimeError("Source exists; use a fresh Kaggle session for exact SHA provenance")
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, str(SOURCE)], check=True)
SOURCE_SHA = subprocess.check_output(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True).strip()
assert len(SOURCE_SHA) == 40
print("BloomML R2 commit:", SOURCE_SHA)


In [ ]:
import sys
sys.path.insert(0, str(SOURCE))
from tools.ml.kaggle_gpu_r2_runner import run
manifest = run(
    output_dir=Path("/kaggle/working/bloomml-run"),
    source_dir=SOURCE, seed=1729,
    scenarios_per_gpu=256, parity_per_gpu=8,
    ticks=8, dt_s=2.0,
)
assert manifest["gpu_physics_ready"] is True
assert len(manifest["devices"]) == 2
print("CUDA PHYSICS:", manifest["simulation_backend"])
for d in manifest["devices"]:
    print(d["device"], d["gpu_name"],
          "scenarios", d["batch_scenarios"],
          "seconds", round(d["batch_wall_seconds"], 3),
          "water residual kg", d["batch_max_water_residual_kg"])
print("Manifest: /kaggle/working/bloomml-run/r2_gpu_physics_manifest.json")


In [ ]:
import json
print(json.dumps({
    "source_commit": manifest["source_commit"],
    "gpu_physics_ready": manifest["gpu_physics_ready"],
    "scenarios_total": manifest["scenarios_total"],
    "calibrated": manifest["calibrated"],
    "devices": [d["device"] for d in manifest["devices"]],
}, indent=2))
